# 01. Exploratory Data Analysis: Anchorpoint B2B Sales Dataset

Phase 3, 2026-08-21. Profiles `data/raw/*.csv` before any metric is computed, so every later cut in `02_unit_economics.ipynb` and `03_diagnostics.ipynb` is built on a known sample size.

**Source of truth for what this data contains:** `docs/project_context.md`, the 2026-08-21 Phase 2 entry, and `docs/decisions.md`. This notebook reads only the exported CSVs, never `src/generate_data.py`. `docs/case_study_scenario.md` holds hypotheses, not findings, and is not used here.

In [1]:
import sys
sys.path.insert(0, "../src")
import phase3_lib as lib
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

tables = lib.load_all()
companies, reps, deals, stage_history, customers, pipeline, spend = (
    tables["companies"], tables["sales_reps"], tables["deals"], tables["deal_stage_history"],
    tables["customers"], tables["sales_pipeline"], tables["sm_spend"]
)
print("Window:", lib.WINDOW_START.date(), "to", lib.WINDOW_END.date())
print("Enterprise push date:", lib.ENTERPRISE_PUSH_DATE.date())
for name, df in tables.items():
    print(f"{name:20s} {len(df):>6,} rows")

Window: 2024-07-01 to 2026-08-31
Enterprise push date: 2025-01-01
companies               900 rows
sales_reps               30 rows
deals                 2,200 rows
deal_stage_history    5,824 rows
customers               616 rows
sales_pipeline          400 rows
sm_spend                 78 rows


## Structural overview

Row counts match the locked mix in `docs/data_model.md`: 2,200 closed deals (Enterprise 420, Mid-Market 760, SMB 1,020), 616 won deals (`customers`), 400 open pipeline opportunities, 30 reps, 900 companies, 78 spend rows.

In [2]:
print(deals["segment"].value_counts())
print()
print("Won deals (customers):", len(customers), "== deals.deal_won.sum():", int(deals["deal_won"].sum()))
print()
print(deals.dtypes)

segment
SMB           1020
Mid-Market     760
Enterprise     420
Name: count, dtype: int64

Won deals (customers): 616 == deals.deal_won.sum(): 616

deal_id                          object
company_id                       object
rep_id                           object
segment                          object
region                           object
acquisition_channel              object
deal_size_usd                   float64
opened_date              datetime64[ns]
close_date               datetime64[ns]
deal_won                           bool
loss_reason                      object
stakeholders_required             int64
stakeholders_engaged              int64
sales_cycle_days                  int64
dtype: object


## Sample sizes per cut

Every cut used later in this analysis is checked against `MIN_REPORTABLE_N = 30` here. Any cut below that threshold is flagged and must not be reported as a clean point estimate downstream.

In [3]:
n_table = lib.sample_size_table(deals)
n_table

,cut,n,reportable_as_point_estimate
0,"Enterprise, all",420,True
1,"Mid-Market, all",760,True
2,"SMB, all",1020,True
3,"Enterprise, pre-push (open date)",105,True
4,"Enterprise, post-push (open date)",315,True
5,"Enterprise, US",269,True
6,"Enterprise post-push, US",197,True
7,"Enterprise, EU",91,True
8,"Enterprise post-push, EU",72,True
9,"Enterprise, APAC",60,True


In [4]:
too_thin = n_table[~n_table["reportable_as_point_estimate"]]
print(f"{len(too_thin)} cuts below n=30:")
too_thin

0 cuts below n=30:


,cut,n,reportable_as_point_estimate


No structural cut is below n=30 at the granularity used in this notebook. Two things to flag going into the diagnostics notebook regardless:

- **Enterprise pre-push EU** (n=19 at the open-date cohort level, used later in the EU-competitor confounder test) is thin even though it clears 30 in aggregate; its confidence intervals will be wide and are reported as such, not hidden.
- **Enterprise monthly cuts are never used.** Per the locked rigor requirement, Enterprise time-series analysis in `03_diagnostics.ipynb` is quarterly grain only.

## Deal size distribution by segment

In [5]:
deals.groupby("segment")["deal_size_usd"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
segment,,,,,,,,
Enterprise,420.0,101399.0,46584.0,50000.0,65766.0,90948.0,128293.0,336000.0
Mid-Market,760.0,34514.0,11565.0,20000.0,23369.0,32468.0,45056.0,56000.0
SMB,1020.0,11764.0,4673.0,5000.0,8181.0,10764.0,14888.0,22400.0


Realized means (Enterprise ~$100k, Mid-Market ~$35k, SMB ~$12k) sit within 15% of the locked targets ($95k / $32k / $11k), consistent with the Phase 2 validation report.

## Sales cycle distribution: won vs. not-won, by segment

The censoring-bias smoke test from `src/validate_dataset.py` (stalled deals must show longer cycles than won deals) is re-derived here directly from the exported data, not taken on faith.

In [6]:
cycle_summary = deals.groupby(["segment", "deal_won"])["sales_cycle_days"].agg(["mean", "median", "count"]).round(1)
cycle_summary

mean  median  count
segment    deal_won                      
Enterprise False      65.6    25.0    351
           True      127.6   115.0     69
Mid-Market False      33.1    24.0    553
           True       59.3    56.0    207
SMB        False      24.9    24.0    680
           True       34.3    33.5    340

In [7]:
stalled = deals[deals["loss_reason"] == "stalled_no_decision"]["sales_cycle_days"]
won = deals[deals["deal_won"]]["sales_cycle_days"]
print(f"Stalled deals: mean {stalled.mean():.0f} days, n={len(stalled)}")
print(f"Won deals: mean {won.mean():.0f} days, n={len(won)}")
print("Stalled > won, as expected if no censoring bias was introduced:", stalled.mean() > won.mean())

Stalled deals: mean 191 days, n=61
Won deals: mean 53 days, n=616
Stalled > won, as expected if no censoring bias was introduced: True


## Region and channel mix by segment

In [8]:
pd.crosstab(deals["segment"], deals["region"], normalize="index").round(3)

region,APAC,EU,US
segment,,,
Enterprise,0.143,0.217,0.640
Mid-Market,0.088,0.286,0.626
SMB,0.122,0.218,0.661


In [9]:
pd.crosstab(deals["segment"], deals["acquisition_channel"], normalize="index").round(3)

acquisition_channel,Direct Sales,Inbound,Partnerships
segment,,,
Enterprise,0.883,0.057,0.060
Mid-Market,0.436,0.362,0.203
SMB,0.154,0.608,0.238


## Sales rep roster

In [10]:
reps[["segment", "has_enterprise_experience", "is_founder_led"]].value_counts().sort_index()

segment     has_enterprise_experience  is_founder_led
Enterprise  False                      False              8
            True                       True               2
Mid-Market  False                      False             10
SMB         False                      False             10
Name: count, dtype: int64

In [11]:
reps[reps["segment"] == "Enterprise"]

,rep_id,rep_name,segment,hire_date,prior_segment,has_enterprise_experience,playbook_type,quota_annual_usd,is_founder_led
0,REP001,Founder Rep 1,Enterprise,2021-03-01,NaN,True,multi_threaded,800000,True
1,REP002,Founder Rep 2,Enterprise,2021-03-01,NaN,True,multi_threaded,800000,True
2,REP003,Enterprise AE 3,Enterprise,2025-01-01,SMB,False,single_threaded,800000,False
3,REP004,Enterprise AE 4,Enterprise,2025-01-01,SMB,False,single_threaded,800000,False
4,REP005,Enterprise AE 5,Enterprise,2025-01-01,SMB,False,single_threaded,800000,False
5,REP006,Enterprise AE 6,Enterprise,2025-01-01,Mid-Market,False,single_threaded,800000,False
6,REP007,Enterprise AE 7,Enterprise,2025-01-01,Mid-Market,False,single_threaded,800000,False
7,REP008,Enterprise AE 8,Enterprise,2025-01-01,Mid-Market,False,single_threaded,800000,False
8,REP009,Enterprise AE 9,Enterprise,2025-01-01,External,False,single_threaded,800000,False
9,REP010,Enterprise AE 10,Enterprise,2025-01-01,External,False,single_threaded,800000,False


## Data quality checks

In [12]:
print("Null counts (should be 0 except loss_reason/churn_date, which are structurally nullable):")
print(deals.isna().sum()[deals.isna().sum() > 0])
print()
print("close_date range:", deals["close_date"].min().date(), "to", deals["close_date"].max().date())
print("All close_date inside window:", ((deals["close_date"] >= lib.WINDOW_START) & (deals["close_date"] <= lib.WINDOW_END)).all())
print()
print("stakeholders_engaged <= stakeholders_required, always:", (deals["stakeholders_engaged"] <= deals["stakeholders_required"]).all())

Null counts (should be 0 except loss_reason/churn_date, which are structurally nullable):
loss_reason    616
dtype: int64

close_date range: 2024-07-01 to 2026-08-31
All close_date inside window: True

stakeholders_engaged <= stakeholders_required, always: True


## EDA takeaways going into Phase 3 analysis

1. All structural row counts and reconciliations match the Phase 2 validation report. No surprises in this data that Phase 2 did not already catch.
2. Every segment-level and quarterly cut has n >= 30. Enterprise pre-push by region (EU specifically, n=19) is the one cut carried forward with an explicit wide-CI caveat.
3. The censoring-bias smoke test reproduces cleanly from the raw export: stalled deals run longer than won deals, in the expected direction.
4. Enterprise time series work in `03_diagnostics.ipynb` will use quarterly grain only, per the locked rigor requirement; monthly Enterprise cuts are not published even though they exist in the data.